# LLM 方法 × Laurent 的 PromptORE 流程（Linked-Re-DocRED）

**这个笔记本做什么：** 把我们的 LLM 方法放进 Laurent 仓库里的 M1 / M2 流程，只换掉"怎么表示一对实体"这一步，其他全部用仓库自己的代码（候选实体对、上下文、训练采样、检测器、分类器、聚类、阈值、评测脚本），所以结果可以直接和 Laurent 的 PromptORE 比。

| 表示 | 含义 |
|---|---|
| `bert` | Laurent 的 PromptORE（BERT `[MASK]`），原样重跑，作为参照 |
| `qwen-state` | 方法 1：Qwen 在开始回答那一刻的内部向量 |
| `qwen-text` | 方法 2：Qwen 写出 `关系 \| 头实体类型 \| 尾实体类型`，再用句向量模型编码 |
| `api-text` | 方法 2.5：同样的提示词，由 DeepSeek 回答 |
| `bert+…` | 混合：BERT 向量拼上文字向量 |

每种表示都跑 **M2（分类器）** 和 **M1（聚类）**；有 DeepSeek 时还会跑"LLM 给簇起名"和"完全不用标签"两种。

**规模：** 训练用共享子集 `train_debug`（300 篇），评测用 `dev_debug`（100 篇）。Laurent 的 BERT 方法在同样的小规模下重跑，所以对比是公平的。

**运行前：**
1. `代码执行程序 → 更改运行时类型 → T4 GPU`
2. 左侧钥匙图标（Secrets）里添加 `DEEPSEEK_API_KEY`，并打开"笔记本访问权限"（没有的话运行时会让你输入一次）
3. `代码执行程序 → 全部运行`。会先弹一次 Google Drive 授权（用来存进度，断线后重新"全部运行"会接着跑，不会重算）

**流程：** 先用几篇文档做一次 3 到 5 分钟的**彩排**（把整条流程走一遍，有问题马上报错），然后才开始正式运行。正式运行在 T4 上大约 2 到 2.5 小时，结束后自动下载 `results_llm_redocred.zip`，把它发给我就行。

In [ ]:
!nvidia-smi -L || echo 'NO GPU: switch the runtime to T4 GPU'
!pip -q install -U sentence-transformers openai

In [ ]:
import os, warnings
os.environ["PYTHONWARNINGS"] = "ignore"        # also for the worker processes of scikit-learn
warnings.filterwarnings("ignore")

## 1. 设置

In [ ]:
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"       # "Qwen/Qwen2.5-1.5B-Instruct" is about twice as fast
ST_NAME  = "sentence-transformers/all-mpnet-base-v2"
K_VALUES = [50, 100, 200, 400]              # number of clusters tried for M1, as in the repository
LLM_BATCH = 32

USE_DEEPSEEK = True
DS_MODEL     = "deepseek-chat"
DS_BASE_URL  = "https://api.deepseek.com"
DS_WORKERS   = 32                            # parallel requests

USE_DRIVE      = True                        # keep the progress on Google Drive (resume after a disconnect)
ONLY_REHEARSAL = False                       # True: stop after the short rehearsal

REPO_URL    = "https://github.com/ZHU-Laurent/Knowledge-graph-extraction-from-text.git"
REPO_COMMIT = "5d3c818823f0cce0c3991cd165d5dad378080b10"   # main on 2026-11-16
DATA_URL    = "https://raw.githubusercontent.com/alteca/Linked-DocRED/main/"

## 2. 函数库（`kgllm.py`）

In [ ]:
%%writefile kgllm.py
"""LLM pair representations inside the PromptORE pipeline of the team repository (methods M1 and M2).

Everything except the representation of a pair is the repository's own code and protocol:
candidates, context ("mention_sentences"), entity names, train sample (all related pairs + 3 unrelated
per related, seed 0, weighted), logistic detector, one logistic classifier per relation (M2), k-means
clusters named by the share of train relations (M1), fact score P(related) x typing score, fact
threshold and k chosen on dev, shared evaluator (scripts/evaluate.py).

Representations of a pair:
  bert        BERT [MASK] state: the repository's PromptORE, unchanged (reference)
  qwen-state  state of a causal LLM at the position where its answer starts         ("method 1")
  qwen-text   the LLM answers "relation | head type | tail type" -> sentence encoder ("method 2")
  api-text    same prompt answered by an API model (DeepSeek)                       ("method 2.5")
  bert+text   bert and the best text representation, concatenated
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import threading
import time

import numpy as np

SEP = "\x1f"


# ============================================================================ resumable cache
class Cache:
    """Runs fn over items in length-sorted chunks and saves each chunk, so a restarted run resumes."""

    def __init__(self, root):
        self.root = root
        os.makedirs(root, exist_ok=True)

    def run(self, name, items, fn, chunk=2048, kind="npy"):
        """kind: "npy" (fn -> array), "json" (fn -> list) or "both" (fn -> (array, list))."""
        if kind == "both":
            return self._both(name, items, fn, chunk)
        return self._run(name, items, fn, chunk, kind)

    def _both(self, name, items, fn, chunk):
        """Chunk by chunk: the array first, then the texts of the same chunk."""
        sig = hashlib.sha1("\x1e".join(items).encode("utf-8")).hexdigest()[:10]
        folder = os.path.join(self.root, f"{name}_{sig}")
        os.makedirs(folder, exist_ok=True)
        order = sorted(range(len(items)), key=lambda i: len(items[i]))
        arrays, texts, t0, done = [None] * len(items), [None] * len(items), time.time(), 0
        n_chunks = (len(order) + chunk - 1) // chunk
        for c in range(n_chunks):
            idx = order[c * chunk:(c + 1) * chunk]
            a_path, t_path = (os.path.join(folder, f"part{c:04d}.{ext}") for ext in ("npy", "json"))
            a = t = None
            if os.path.exists(a_path) and os.path.exists(t_path):
                try:
                    a, t = np.load(a_path).astype(np.float32), json.load(open(t_path, encoding="utf-8"))
                except Exception:
                    a = t = None
                if a is not None and (len(a) != len(idx) or len(t) != len(idx)):
                    a = t = None
            if a is None:
                batch = [items[i] for i in idx]
                a, t = fn(batch)
                a, t = np.asarray(a), list(t)
                if len(a) != len(idx) or len(t) != len(idx):
                    raise ValueError(f"{name}: wrong number of results")
                np.save(a_path + ".tmp.npy", a.astype(np.float16))
                with open(t_path + ".tmp", "w", encoding="utf-8") as f:
                    json.dump(t, f)
                os.replace(t_path + ".tmp", t_path)
                os.replace(a_path + ".tmp.npy", a_path)
                a = a.astype(np.float16).astype(np.float32)
                done += len(idx)
                rate = done / max(time.time() - t0, 1e-6)
                left = len(order) - (c + 1) * chunk
                print(f"  {name}: chunk {c + 1}/{n_chunks}  {rate:.0f} items/s  about "
                      f"{max(left, 0) / max(rate, 1e-6) / 60:.0f} min left", flush=True)
            for i, row, text in zip(idx, a, t):
                arrays[i], texts[i] = row, text
        return np.stack(arrays).astype(np.float32), texts

    def _run(self, name, items, fn, chunk=2048, kind="npy"):
        sig = hashlib.sha1("\x1e".join(items).encode("utf-8")).hexdigest()[:10]
        folder = os.path.join(self.root, f"{name}_{sig}")
        os.makedirs(folder, exist_ok=True)
        order = sorted(range(len(items)), key=lambda i: len(items[i]))
        out, t0, done = [None] * len(items), time.time(), 0
        n_chunks = (len(order) + chunk - 1) // chunk
        for c in range(n_chunks):
            idx = order[c * chunk:(c + 1) * chunk]
            path = os.path.join(folder, f"part{c:04d}.{kind}")
            res = None
            if os.path.exists(path):
                try:
                    res = np.load(path).astype(np.float32) if kind == "npy" else json.load(open(path, encoding="utf-8"))
                except Exception:
                    res = None
                if res is not None and len(res) != len(idx):
                    res = None
            if res is None:
                res = fn([items[i] for i in idx])
                if len(res) != len(idx):
                    raise ValueError(f"{name}: {len(res)} results for {len(idx)} items")
                tmp = os.path.join(folder, f"tmp{c:04d}.{kind}")
                if kind == "npy":
                    np.save(tmp, np.asarray(res).astype(np.float16))
                    res = np.asarray(res).astype(np.float16).astype(np.float32)
                else:
                    with open(tmp, "w", encoding="utf-8") as f:
                        json.dump(list(res), f)
                os.replace(tmp, path)
                done += len(idx)
                rate = done / max(time.time() - t0, 1e-6)
                left = sum(len(order[k * chunk:(k + 1) * chunk]) for k in range(c + 1, n_chunks))
                print(f"  {name}: chunk {c + 1}/{n_chunks}  {rate:.0f} items/s  about {left / max(rate, 1e-6) / 60:.0f} min left",
                      flush=True)
            for i, r in zip(idx, res):
                out[i] = r
        return np.stack(out).astype(np.float32) if kind == "npy" else out


def inst_key(inst):
    return SEP.join((inst["context"], inst["head"], inst["tail"]))


def key_inst(key):
    context, head, tail = key.split(SEP)
    return {"context": context, "head": head, "tail": tail}


# ============================================================================ local causal LLM
STRUCT_SYSTEM = (
    'You label the relation between two entities mentioned in a text.\n'
    'Answer with one line, in exactly this format and with nothing else:\n'
    '<relation> | <head type> | <tail type>\n'
    '<relation> is a short phrase (1 to 5 words) for the relation read from the head entity to the tail '
    'entity, or the word none when the text states no relation from the head entity to the tail entity. '
    'Many pairs of entities have no relation.\n'
    '<head type> and <tail type> are the general types of the two entities (1 to 3 words each).\n'
    'Use only what the text says.')
STRUCT_USER = 'Text: "{context}"\nHead entity: "{head}"\nTail entity: "{tail}"'


def struct_user(inst):
    return STRUCT_USER.format(**inst)


def _oom(e):
    return "out of memory" in str(e).lower()


class CausalLM:
    """One loaded LLM for the last-token embeddings (C) and the generated answers (D2)."""

    def __init__(self, model_name, device="cuda"):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.tok.padding_side = "left"            # the last position is the last real token
        self.tok.truncation_side = "left"         # never cut the end of the prompt
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dtype = torch.float16 if device == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device).eval()
        self.device = device

    def _batched(self, texts, bs, step):
        """Apply step to batches; on a GPU out-of-memory error the batch size is halved and retried."""
        out, i = [], 0
        while i < len(texts):
            try:
                out.extend(step(texts[i:i + bs]))
                i += bs
            except RuntimeError as e:
                if not _oom(e) or bs == 1:
                    raise
                self.torch.cuda.empty_cache()
                bs = max(1, bs // 2)
                print(f"  out of memory, batch size -> {bs}", flush=True)
        return out

    def answer_with_state(self, users, system=None, bs=32, max_new_tokens=24, max_len=768):
        """One pass per prompt: the generated answer, and the last-layer state at the last token of the
        prompt, the position from which the model starts to write its answer.

        -> (states (n, hidden), answers). The state is read at the input of the output layer during the
        first step of the generation, so the prompt is only encoded once."""
        torch = self.torch
        prompts = []
        for u in users:
            msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": u}]
            prompts.append(self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        captured = []

        def hook(module, args):
            if not captured:                        # first call = the whole prompt
                captured.append(args[0][:, -1, :].detach().float().cpu().numpy())
        handle = self.model.get_output_embeddings().register_forward_pre_hook(hook)

        def step(batch):
            captured.clear()
            enc = self.tok(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=max_len).to(self.device)
            with torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                          pad_token_id=self.tok.pad_token_id)
            texts = self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            if not captured or len(captured[0]) != len(batch):
                raise ValueError("the state of the prompt was not captured")
            return list(zip(captured[0], texts))
        try:
            out = self._batched(prompts, bs, step)
        finally:
            handle.remove()
        return np.stack([state for state, _ in out]), [text for _, text in out]


# ============================================================================ parsing the answers
def clean_phrase(p):
    p = p.strip().split("\n")[0].strip().strip('"\'`.:;,*<> ').strip().lower()
    return " ".join(re.sub(r"\s+", " ", p).split()[:6])


NONE_WORDS = {"", "none", "no relation", "n/a", "na", "null", "unknown", "no", "nothing", "not stated",
              "no relationship", "none stated", "unrelated"}


def parse_struct(text):
    """-> (relation, head type, tail type). relation is "none" when the model states no relation."""
    lines = [line for line in (text or "").splitlines() if line.strip()]
    parts = [clean_phrase(re.sub(r"^\s*(relation|answer)\s*[:：]", "", part, flags=re.I))
             for part in (lines[0] if lines else "").split("|")]
    parts += [""] * (3 - len(parts))
    rel = "none" if parts[0] in NONE_WORDS else parts[0]
    return rel, parts[1] or "entity", parts[2] or "entity"


def typed_text(struct):
    rel, head_type, tail_type = struct
    return f"{head_type} | {rel} | {tail_type}"


# ============================================================================ API model (DeepSeek)
class APIChat:
    """OpenAI-compatible chat client with a thread pool, retries and an append-only cache on disk.

    The cache keeps one line per answered prompt, so an interrupted run resumes and the raw answers
    are kept (rule 5 of docs/collaboration.md)."""

    def __init__(self, api_key, model, base_url, cache_path, workers=32):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key, base_url=base_url, timeout=120, max_retries=0)
        self.model, self.workers, self.cache_path = model, workers, cache_path
        self.lock = threading.Lock()
        self.usage = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "cached_prompt_tokens": 0,
                      "failed": 0}
        self.cache = {}
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        if os.path.exists(cache_path):
            with open(cache_path, encoding="utf-8") as f:
                for line in f:
                    try:
                        row = json.loads(line)
                        self.cache[row["k"]] = row["a"]
                    except Exception:
                        pass                                  # a line cut by an interrupted write
        self._file = open(cache_path, "a", encoding="utf-8")

    def _key(self, system, user, max_tokens):
        return hashlib.sha1(f"{self.model}|{max_tokens}|{system}|{user}".encode("utf-8")).hexdigest()

    def chat(self, user, system=None, max_tokens=60, retries=6):
        key = self._key(system, user, max_tokens)
        if key in self.cache:
            return self.cache[key]
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        for attempt in range(retries):
            try:
                r = self.client.chat.completions.create(model=self.model, messages=messages,
                                                        max_tokens=max_tokens, temperature=0.0)
                text = r.choices[0].message.content or ""
                u = getattr(r, "usage", None)
                with self.lock:
                    self.usage["calls"] += 1
                    if u is not None:
                        self.usage["prompt_tokens"] += getattr(u, "prompt_tokens", 0) or 0
                        self.usage["completion_tokens"] += getattr(u, "completion_tokens", 0) or 0
                        self.usage["cached_prompt_tokens"] += getattr(u, "prompt_cache_hit_tokens", 0) or 0
                    self.cache[key] = text
                    self._file.write(json.dumps({"k": key, "a": text}, ensure_ascii=False) + "\n")
                    if self.usage["calls"] % 200 == 0:
                        self._file.flush()
                return text
            except Exception as e:
                status = getattr(e, "status_code", None)
                if status in (400, 401, 402, 403, 404, 422):      # will not succeed on a retry
                    with self.lock:
                        self.usage["failed"] += 1
                        self.usage.setdefault("errors", {})
                        name = f"{status} {type(e).__name__}"
                        self.usage["errors"][name] = self.usage["errors"].get(name, 0) + 1
                        self.usage.setdefault("last_error", str(e)[:300])
                    return ""
                time.sleep(min(2 ** attempt, 30))
        with self.lock:
            self.usage["failed"] += 1
        return ""

    def missing(self, users, system=None, max_tokens=60):
        """Number of prompts without an answer (failed calls are not cached, so map() retries them)."""
        return sum(self._key(system, u, max_tokens) not in self.cache for u in users)

    def map(self, users, system=None, max_tokens=60, status=None):
        from concurrent.futures import ThreadPoolExecutor, as_completed
        out = [None] * len(users)
        status = status if status is not None else {}
        status.update({"total": len(users), "done": 0})
        with ThreadPoolExecutor(self.workers) as pool:
            futures = {pool.submit(self.chat, u, system, max_tokens): i for i, u in enumerate(users)}
            for f in as_completed(futures):
                out[futures[f]] = f.result()
                status["done"] += 1
        self._file.flush()
        return out


class Background:
    """Runs fn(status) in a thread, so the API calls go on while the GPU works."""

    def __init__(self, fn):
        self.status, self.result, self.error = {"total": 0, "done": 0}, None, None

        def target():
            try:
                self.result = fn(self.status)
            except BaseException as e:               # shown when the result is asked for
                self.error = e
        self.thread = threading.Thread(target=target, daemon=True)
        self.t0 = time.time()
        self.thread.start()

    def wait(self, every=60):
        while self.thread.is_alive():
            self.thread.join(every)
            done, total = self.status.get("done", 0), self.status.get("total", 0)
            rate = done / max(time.time() - self.t0, 1e-6)
            if self.thread.is_alive():
                print(f"  API: {done}/{total} answers, about {(total - done) / max(rate, 1e-6) / 60:.0f} min left",
                      flush=True)
        if self.error is not None:
            raise self.error
        return self.result


# ============================================================================ representations
def fit_normalizer(X_train):
    """Centre on the train mean, then L2-normalise (tames the few huge dimensions of LLM states)."""
    mean = X_train.mean(axis=0, keepdims=True)

    def apply(X):
        X = X - mean
        return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    return apply


def concat(*parts):
    return (np.concatenate(parts, axis=1) / np.sqrt(len(parts))).astype(np.float32)


class TextEncoder:
    def __init__(self, model_name, device):
        from sentence_transformers import SentenceTransformer
        self.model, self.lut = SentenceTransformer(model_name, device=device), {}

    def __call__(self, texts):
        new = sorted({t for t in texts if t not in self.lut})
        if new:
            vectors = self.model.encode(new, batch_size=256, normalize_embeddings=True, show_progress_bar=False)
            self.lut.update(zip(new, np.asarray(vectors, dtype=np.float32)))
        return np.stack([self.lut[t] for t in texts])


# ============================================================================ the repository's M2 and M1
def micro_f1(facts, gold):
    correct = len(gold & set(facts))
    return 2 * correct / (len(facts) + len(gold)) if facts and gold else 0.0


def usable_k(k_values, X):
    """k-means needs clearly more distinct points than clusters (generated phrases repeat a lot)."""
    n_distinct = len(np.unique(np.round(X, 4), axis=0)) if len(X) < 50000 else len(X)
    return [k for k in k_values if k <= n_distinct // 2] or [max(2, min(min(k_values), n_distinct // 2))]


def run_m2(X_train, labels, negative_weight, X_dev, dev_pairs, relations, gold):
    """M2 of the repository: detector x one classifier per relation, threshold chosen on dev."""
    from kg_extraction.relations.probe import RelationProbe, best_threshold, facts_above
    probe = RelationProbe(relations).fit(X_train, labels, negative_weight)
    scores = probe.scores(X_dev)
    threshold = best_threshold(dev_pairs, scores, relations, gold)
    facts = facts_above(dev_pairs, scores, relations, threshold)
    return probe, facts, {"threshold": round(threshold, 2), "dev_f1": round(100 * micro_f1(facts, gold), 2)}


def run_m1(probe, X_train, labels, X_dev, dev_pairs, relations, gold, k_values, seed=0):
    """M1 of the repository: same detector, k-means on related train pairs, k and threshold from dev."""
    from kg_extraction.relations.clustering import ClusterTyper
    from kg_extraction.relations.probe import best_threshold, facts_above
    related = np.array([bool(codes) for codes in labels])
    related_labels = [codes for codes in labels if codes]
    detection = probe.detector.predict_proba(X_dev)[:, 1]
    best, tried = None, {}
    for k in usable_k(k_values, X_train[related]):
        typer = ClusterTyper(relations, k, seed).fit(X_train[related], related_labels)
        scores = detection[:, None] * typer.scores(X_dev)
        threshold = best_threshold(dev_pairs, scores, relations, gold)
        facts = facts_above(dev_pairs, scores, relations, threshold)
        f1 = micro_f1(facts, gold)
        tried[k] = round(100 * f1, 2)
        if best is None or f1 > best[0]:
            best = (f1, facts, typer, {"k": k, "threshold": round(threshold, 2)})
    _, facts, typer, settings = best
    settings.update({"dev_f1_by_k": tried, "dev_f1": tried[settings["k"]]})
    return typer, detection, facts, settings


def thresholded(scores, dev_pairs, relations, gold):
    from kg_extraction.relations.probe import best_threshold, facts_above
    threshold = best_threshold(dev_pairs, scores, relations, gold)
    facts = facts_above(dev_pairs, scores, relations, threshold)
    return facts, {"threshold": round(threshold, 2), "dev_f1": round(100 * micro_f1(facts, gold), 2)}


# ============================================================================ naming clusters with an LLM
NAME_SYSTEM = ('You are given pairs of entities that were grouped together automatically because the texts '
               'seem to express the same relation between the head entity and the tail entity. '
               'Choose the one Wikidata property of the list that best describes the relation from head to '
               'tail shared by most pairs. Answer with the property code only (for example P17). '
               'Answer NONE if no property of the list fits.\n\nProperties:\n{relations}')


def name_system(relation_names):
    lines = [f"{code}: {name}" for code, name in sorted(relation_names.items(), key=lambda kv: int(kv[0][1:]))]
    return NAME_SYSTEM.format(relations="\n".join(lines))


def _short(text, n=45):
    words = text.split()
    return text if len(words) <= n else " ".join(words[:n]) + " ..."


def naming_prompts(clusters, instances, n_clusters, n_examples=8, seed=0):
    """One prompt per cluster, from a random sample of its members. No label is shown."""
    rng = np.random.RandomState(seed)
    clusters = np.asarray(clusters)
    prompts = []
    for c in range(n_clusters):
        members = np.where(clusters == c)[0]
        if len(members) == 0:
            prompts.append(None)
            continue
        picked = rng.choice(members, size=min(n_examples, len(members)), replace=False)
        lines = [f'- head: "{instances[i]["head"]}" ; tail: "{instances[i]["tail"]}" ; text: "{_short(instances[i]["context"])}"'
                 for i in picked]
        prompts.append("Pairs of the group:\n" + "\n".join(lines))
    return prompts


def parse_code(answer, relations):
    for code in re.findall(r"P\d+", answer or ""):
        if code in relations:
            return code
    return None


def name_clusters(chat, clusters, instances, n_clusters, relation_names):
    """-> list with the relation code of each cluster, or None (empty cluster, NONE, or unreadable)."""
    prompts = naming_prompts(clusters, instances, n_clusters)
    system = name_system(relation_names)
    asked = [i for i, p in enumerate(prompts) if p is not None]
    answers = chat.map([prompts[i] for i in asked], system=system, max_tokens=8)
    codes = [None] * n_clusters
    for i, answer in zip(asked, answers):
        codes[i] = parse_code(answer, relation_names)
    return codes


def one_hot_names(codes, relations):
    position = {code: column for column, code in enumerate(relations)}
    matrix = np.zeros((len(codes), len(relations)))
    for cluster, code in enumerate(codes):
        if code is not None:
            matrix[cluster, position[code]] = 1.0
    return matrix


# ============================================================================ run folders
def write_run(directory, card, docs, facts, subset, seed=0, clusters=None):
    """Write a run folder in the format of docs/collaboration.md.

    facts: {(doc index, head, tail, relation): score}; clusters: {(doc index, head, tail): cluster}."""
    os.makedirs(directory, exist_ok=True)
    rows = sorted((str(docs[i]["id"]), h, r, t, round(float(s), 6)) for (i, h, t, r), s in facts.items())
    with open(os.path.join(directory, f"redocred_{subset}_facts_seed{seed}.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["doc_id", "h", "r", "t", "score"])
        w.writerows(rows)
    if clusters is not None:
        with open(os.path.join(directory, f"redocred_{subset}_clusters_seed{seed}.csv"), "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["doc_id", "h", "t", "cluster"])
            w.writerows(sorted((str(docs[i]["id"]), h, t, int(c)) for (i, h, t), c in clusters.items()))
    with open(os.path.join(directory, "card.json"), "w", encoding="utf-8") as f:
        json.dump(card, f, indent=1)


# ============================================================================ the whole experiment
def subset_docs(docs, ids, limit=None):
    """Documents of a fixed subset of the benchmark, in the order of the split."""
    ids = set(ids)
    chosen = [doc for doc in docs if str(doc["id"]) in ids]
    return chosen[:limit] if limit else chosen


def bert_embeddings(cache_dir, instances, device):
    """The repository's PromptORE embeddings (relations/encoder.py, unchanged), kept on disk."""
    from kg_extraction.relations.encoder import embed_instances
    sig = hashlib.sha1("\x1e".join(inst_key(i) for i in instances).encode("utf-8")).hexdigest()[:10]
    path = os.path.join(cache_dir, f"bert_{sig}.npy")
    if os.path.exists(path):
        return np.load(path).astype(np.float32)
    embeddings, n_truncated = embed_instances(instances, device=device)
    if n_truncated:
        print(f"  {n_truncated} contexts truncated", flush=True)
    np.save(path + ".tmp.npy", embeddings.astype(np.float16))
    os.replace(path + ".tmp.npy", path)
    return embeddings


def evaluate(repo_dir, run_dir, leaderboard):
    """Score a run folder with the shared evaluator of the repository."""
    import subprocess
    import sys
    done = subprocess.run([sys.executable, "scripts/evaluate.py", os.path.abspath(run_dir),
                           "--leaderboard", os.path.abspath(leaderboard)],
                          cwd=repo_dir, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"evaluate.py failed on {run_dir}:\n{done.stdout}\n{done.stderr}")
    line = [l for l in done.stdout.splitlines() if l.startswith("redocred_")]
    print(f"    {os.path.basename(run_dir):38s} {line[0].split('seeds=1')[-1].strip() if line else ''}", flush=True)


def run_experiment(repo_dir, out_dir, cache_dir, train_docs, dev_docs, allowed, relation_names,
                   llm, llm_name, encoder, encoder_name, chat=None, api_model=None,
                   k_values=(50, 100, 200, 400), label_free_k=100, device="cuda",
                   author="Axel", training_data="train_debug (300 docs)", subset="dev_debug",
                   llm_batch=32, api_max_tokens=40):
    """Run every representation through M2 and M1, write the run folders and score them.

    train_docs / dev_docs: documents of train_debug / dev_debug. allowed: the type filter of the whole
    train split, as in the repository. chat: an APIChat, or None to skip the API methods."""
    from sklearn.cluster import KMeans
    from kg_extraction.evaluation.relations import gold_facts
    from kg_extraction.relations.probe import pair_instances, pair_list, sample_train_pairs

    os.makedirs(out_dir, exist_ok=True)
    cache = Cache(cache_dir)
    leaderboard = os.path.join(out_dir, "leaderboard.csv")
    relations = sorted(relation_names)
    times, settings = {}, {}

    # ---- pairs, exactly as scripts/compare_methods.py and scripts/label_efficiency.py
    all_pairs = pair_list(train_docs, allowed)
    kept, negative_weight = sample_train_pairs(all_pairs, 3, 0)
    train_pairs = [all_pairs[i] for i in kept]
    dev_pairs = pair_list(dev_docs, allowed)
    train_inst, dev_inst = pair_instances(train_docs, train_pairs), pair_instances(dev_docs, dev_pairs)
    labels = [pair[3] for pair in train_pairs]
    related = np.array([bool(codes) for codes in labels])
    gold = gold_facts(dev_docs)
    n = len(train_inst)
    instances = train_inst + dev_inst
    users = [struct_user(inst) for inst in instances]
    print(f"train: {len(train_docs)} documents, {len(all_pairs)} candidate pairs, sample of {n} "
          f"({int(related.sum())} related, weight of an unrelated pair {negative_weight:.2f})\n"
          f"dev:   {len(dev_docs)} documents, {len(dev_pairs)} candidate pairs, {len(gold)} gold facts", flush=True)
    settings["data"] = {"train_docs": len(train_docs), "train_sample": n, "train_related": int(related.sum()),
                        "negative_weight": round(float(negative_weight), 3), "dev_docs": len(dev_docs),
                        "dev_pairs": len(dev_pairs), "dev_gold_facts": len(gold)}

    # ---- the API model answers in the background while the GPU works
    background = None
    if chat is not None:
        background = Background(lambda status: chat.map(users, STRUCT_SYSTEM, api_max_tokens, status))

    # ---- representations
    t0 = time.time()
    print("\n[1] BERT [MASK] embeddings (the repository's PromptORE)", flush=True)
    bert = bert_embeddings(cache_dir, instances, device)
    times["bert"] = time.time() - t0

    t0 = time.time()
    print(f"\n[2] {llm_name}: state and answer of every pair", flush=True)
    states, llm_raw = cache.run(
        "llm", users, lambda batch: llm.answer_with_state(batch, STRUCT_SYSTEM, bs=llm_batch), kind="both")
    times["llm"] = time.time() - t0
    if not np.isfinite(states).all():
        raise ValueError("the LLM states contain NaN or infinite values (try another LLM_NAME)")
    llm_struct = [parse_struct(answer) for answer in llm_raw]

    api_raw = api_struct = None
    if chat is not None:
        t0 = time.time()
        print(f"\n[3] {api_model}: waiting for the answers still missing", flush=True)
        api_raw = background.wait()
        for _ in range(2):                                   # failed calls are tried again
            if chat.missing(users, STRUCT_SYSTEM, api_max_tokens):
                api_raw = chat.map(users, STRUCT_SYSTEM, api_max_tokens)
        missing = chat.missing(users, STRUCT_SYSTEM, api_max_tokens)
        print(f"  usage: {chat.usage}", flush=True)
        if missing > 0.02 * len(users):
            raise RuntimeError(f"{missing} of {len(users)} API calls failed: {chat.usage}. "
                               "Run the cell again: the answers already received are kept.")
        api_struct = [parse_struct(answer) for answer in api_raw]
        times["api_wait_after_gpu"] = time.time() - t0
        settings["api"] = {"missing": missing, **{k: v for k, v in chat.usage.items()}}

    # the answers, readable (rule 5 of docs/collaboration.md: keep what the LLM wrote)
    for split, docs, pairs, offset in (("train", train_docs, train_pairs, 0), ("dev", dev_docs, dev_pairs, n)):
        with open(os.path.join(out_dir, f"llm_answers_{split}.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["doc_id", "h", "t", "head", "tail", "gold", "local_llm", "api_llm"])
            for row, (index, head, tail, codes) in enumerate(pairs):
                inst = instances[offset + row]
                w.writerow([docs[index]["id"], head, tail, inst["head"], inst["tail"], " ".join(codes),
                            llm_raw[offset + row].strip().replace("\n", " / "),
                            api_raw[offset + row].strip().replace("\n", " / ") if api_raw else ""])

    def none_share(structs):
        return round(float(np.mean([s[0] == "none" for s in structs])), 3)
    settings["answers"] = {"local_none_share": none_share(llm_struct),
                           "local_distinct_relations": len({s[0] for s in llm_struct})}
    if api_struct:
        settings["answers"].update({"api_none_share": none_share(api_struct),
                                    "api_distinct_relations": len({s[0] for s in api_struct})})
    print(f"\nanswers: {settings['answers']}", flush=True)

    reps = {"bert": (bert[:n], bert[n:], "bert-base-uncased",
                     "PromptORE prompt '{head} [MASK] {tail}.', BERT [MASK] state, the repository's code unchanged")}
    norm = fit_normalizer(states[:n])
    reps["qwen-state"] = (norm(states[:n]), norm(states[n:]), llm_name,
                          "last-layer state of the LLM where its answer starts, centred on train and L2-normalised")
    texts = {"qwen-text": encoder([typed_text(s) for s in llm_struct])}
    reps["qwen-text"] = (texts["qwen-text"][:n], texts["qwen-text"][n:], f"{llm_name}, {encoder_name}",
                         "the LLM writes 'relation | head type | tail type' (relation may be none), "
                         "the answer is embedded by a sentence encoder")
    best_text = "qwen-text"
    if api_struct:
        texts["api-text"] = encoder([typed_text(s) for s in api_struct])
        reps["api-text"] = (texts["api-text"][:n], texts["api-text"][n:], f"{api_model}, {encoder_name}",
                            "an API LLM writes 'relation | head type | tail type' (relation may be none), "
                            "the answer is embedded by a sentence encoder")
        best_text = "api-text"
    bert_norm = fit_normalizer(bert[:n])
    hybrid = concat(bert_norm(bert), texts[best_text])
    reps[f"bert+{best_text}"] = (hybrid[:n], hybrid[n:], f"bert-base-uncased, {reps[best_text][2]}",
                                 f"BERT [MASK] state (centred, L2) concatenated with the {best_text} vector")

    def card(run, method, supervision, models, description, data=training_data):
        return {"run": run, "author": author, "method": method, "supervision": supervision,
                "training_data": data, "models": models, "description": description}

    def save(run, facts, clusters=None, **fields):
        directory = os.path.join(out_dir, run)
        write_run(directory, card(run, **fields), dev_docs, facts, subset, clusters=clusters)
        evaluate(repo_dir, directory, leaderboard)

    protocol = (f"Detector and typing fitted on {training_data}: all related pairs + 3 unrelated per related "
                f"(seed 0, weighted). Threshold chosen on {subset}.")
    related_inst = [inst for inst, codes in zip(train_inst, labels) if codes]

    # ---- M2 and M1 on every representation
    print("\n[4] M2 (classifier) and M1 (clusters) on every representation", flush=True)
    for name, (X_train, X_dev, models, description) in reps.items():
        t0 = time.time()
        print(f"  {name} ({X_train.shape[1]} dimensions)", flush=True)
        probe, facts, m2 = run_m2(X_train, labels, negative_weight, X_dev, dev_pairs, relations, gold)
        save(f"td-{name}-classifier", facts, method=f"{name} + classifier (M2)", supervision="supervised",
             models=models, description=f"{description}. {protocol} Settings: {m2}")

        typer, detection, facts, m1 = run_m1(probe, X_train, labels, X_dev, dev_pairs, relations, gold, k_values)
        assigned = typer.kmeans.predict(X_dev)
        clusters = {(pair[0], pair[1], pair[2]): c for pair, c in zip(dev_pairs, assigned)}
        save(f"td-{name}-clusters", facts, clusters=clusters, method=f"{name} + k-means (M1)",
             supervision="labels_for_mapping", models=models,
             description=f"{description}. {protocol} k chosen on {subset}. Settings: {m1}")
        settings[name] = {"M2": m2, "M1": m1}

        # one relation per cluster: from the train labels (control) or chosen by the API LLM
        k = typer.kmeans.n_clusters
        majority = [relations[column] if typer.precision[c].sum() > 0 else None
                    for c, column in enumerate(typer.precision.argmax(axis=1))]
        _, control = thresholded(detection[:, None] * one_hot_names(majority, relations)[assigned],
                                 dev_pairs, relations, gold)
        settings[name]["M1_one_relation_per_cluster_from_labels"] = control
        if chat is not None:
            codes = name_clusters(chat, typer.kmeans.labels_, related_inst, k, relation_names)
            facts, named = thresholded(detection[:, None] * one_hot_names(codes, relations)[assigned],
                                       dev_pairs, relations, gold)
            named["clusters_named_like_majority_label"] = round(
                float(np.mean([a == b for a, b in zip(codes, majority)])), 3)
            named["clusters_without_name"] = sum(code is None for code in codes)
            save(f"td-{name}-clusters-llmnamed", facts, clusters=clusters,
                 method=f"{name} + k-means, clusters named by an LLM", supervision="labels_for_mapping",
                 models=f"{models}, {api_model}",
                 description=f"{description}. Same detector and clusters as td-{name}-clusters (k={k}), but "
                             f"each cluster gets one relation chosen by {api_model} from 8 of its pairs, without "
                             f"labels. Train labels are only used by the detector. Settings: {named}")
            settings[name]["M1_named_by_llm"] = named
        times[f"fit_{name}"] = time.time() - t0

    # ---- no label at all: the LLM detects (answer is not none), k-means on dev, the API LLM names the clusters
    if chat is not None:
        print("\n[5] Without any label", flush=True)
        for name, structs in (("api-text", api_struct), ("qwen-text", llm_struct)):
            mask = np.array([s[0] != "none" for s in structs[n:]])
            if mask.sum() < 4:
                print(f"  {name}: the LLM finds almost no relation, skipped", flush=True)
                continue
            X = texts[name][n:][mask]
            k = usable_k([label_free_k], X)[0]
            kmeans = KMeans(init="k-means++", n_init=3, n_clusters=k, random_state=0).fit(X)
            picked = [i for i in range(len(dev_pairs)) if mask[i]]
            codes = name_clusters(chat, kmeans.labels_, [dev_inst[i] for i in picked], k, relation_names)
            facts = {(dev_pairs[i][0], dev_pairs[i][1], dev_pairs[i][2], codes[c]): 1.0
                     for i, c in zip(picked, kmeans.labels_) if codes[c] is not None}
            clusters = {(dev_pairs[i][0], dev_pairs[i][1], dev_pairs[i][2]): c for i, c in zip(picked, kmeans.labels_)}
            info = {"k": k, "pairs_called_related": int(mask.sum()),
                    "clusters_without_name": sum(code is None for code in codes),
                    "dev_f1": round(100 * micro_f1(facts, gold), 2)}
            save(f"td-{name}-labelfree", facts, clusters=clusters, data="none",
                 method=f"{name}: LLM detection + k-means + clusters named by an LLM", supervision="none",
                 models=f"{reps[name][2]}, {api_model}",
                 description=f"No label. A pair is related when the LLM answer is not none. k-means (k={k}, fixed) "
                             f"on the answers of the {subset} pairs called related, each cluster named by "
                             f"{api_model} with one of the 96 relations. Settings: {info}")
            settings[f"{name}-labelfree"] = info

    settings["seconds"] = {k: round(v) for k, v in times.items()}
    with open(os.path.join(out_dir, "settings.json"), "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=1)
    return settings


## 3. DeepSeek 密钥（开头问一次，并马上试一次调用）

In [ ]:
DS_KEY = None
if USE_DEEPSEEK:
    try:
        from google.colab import userdata
        DS_KEY = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        pass
    if not DS_KEY:
        from getpass import getpass
        DS_KEY = getpass("DeepSeek API key (leave empty to skip the DeepSeek methods): ").strip() or None
USE_DEEPSEEK = bool(DS_KEY)
if USE_DEEPSEEK:
    import sys
    sys.path.insert(0, os.getcwd())
    import kgllm
    from openai import OpenAI
    _example = {"context": "Marie Curie was born in Warsaw in 1867.", "head": "Marie Curie", "tail": "Warsaw"}
    try:
        _r = OpenAI(api_key=DS_KEY, base_url=DS_BASE_URL, timeout=60).chat.completions.create(
            model=DS_MODEL, max_tokens=40, temperature=0.0,
            messages=[{"role": "system", "content": kgllm.STRUCT_SYSTEM},
                      {"role": "user", "content": kgllm.struct_user(_example)}])
    except Exception as e:
        raise RuntimeError("The DeepSeek test call failed (wrong key, no credit, or wrong model name). "
                           "Fix it, or set USE_DEEPSEEK = False in the settings cell.") from e
    _answer = _r.choices[0].message.content or ""
    print("DeepSeek test call:", repr(_answer), "->", kgllm.parse_struct(_answer))
    if "|" not in _answer:
        raise RuntimeError(f"Unexpected DeepSeek answer {_answer!r} for model {DS_MODEL}: check DS_MODEL "
                           "(it must be a chat model that answers directly), or set USE_DEEPSEEK = False.")
print("DeepSeek methods:", "ON" if USE_DEEPSEEK else "OFF (skipped)")

## 4. 进度保存位置

In [ ]:
import os
CACHE = "/content/cache"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CACHE = "/content/drive/MyDrive/kg_llm_cache"
    except Exception as e:
        print("Google Drive not mounted, the progress is kept only in this session:", e)
os.makedirs(CACHE, exist_ok=True)
print("cache:", CACHE)

## 5. Laurent 的仓库（固定在一个版本）和数据

In [ ]:
import json, subprocess, sys, urllib.request
REPO = "/content/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", REPO_COMMIT], check=True)
DATA = f"{REPO}/data/raw/Linked-Re-DocRED"
os.makedirs(DATA, exist_ok=True)
for name, folder in (("train_revised.json", "Linked-Re-DocRED"), ("dev_revised.json", "Linked-Re-DocRED"),
                     ("test_revised.json", "Linked-Re-DocRED"), ("rel_info.json", "Linked-DocRED")):
    if not os.path.exists(f"{DATA}/{name}"):
        urllib.request.urlretrieve(f"{DATA_URL}{folder}/{name}", f"{DATA}/{name}")
print(subprocess.run([sys.executable, "scripts/prepare_benchmark.py"], cwd=REPO, capture_output=True,
                     text=True, check=True).stdout)
sys.path.insert(0, f"{REPO}/src")
sys.path.insert(0, os.getcwd())

from kg_extraction.data.linked_docred import load_relation_names, load_split
from kg_extraction.pairs.candidates import allowed_type_pairs
import kgllm

relation_names = load_relation_names(DATA)
train_all, dev_all = load_split(DATA, "train"), load_split(DATA, "dev")
ALLOWED = allowed_type_pairs(train_all)              # type filter from the whole train split, as in the repository
subsets = json.load(open(f"{REPO}/data/processed/benchmark/redocred_subsets.json"))
TRAIN_DOCS = kgllm.subset_docs(train_all, subsets["train_debug"]["doc_ids"])
DEV_DOCS = kgllm.subset_docs(dev_all, subsets["dev_debug"]["doc_ids"])
print(len(relation_names), "relations,", len(TRAIN_DOCS), "train_debug documents,", len(DEV_DOCS), "dev_debug documents")

## 6. 加载模型

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
llm = kgllm.CausalLM(LLM_NAME, device=DEVICE)
encoder = kgllm.TextEncoder(ST_NAME, DEVICE)
chat = kgllm.APIChat(DS_KEY, DS_MODEL, DS_BASE_URL, f"{CACHE}/deepseek_answers.jsonl", DS_WORKERS) if USE_DEEPSEEK else None

def run(train_docs, dev_docs, out_dir, cache_dir, **kw):
    return kgllm.run_experiment(REPO, out_dir, cache_dir, train_docs, dev_docs, ALLOWED, relation_names,
                                llm, LLM_NAME, encoder, ST_NAME, chat=chat, api_model=DS_MODEL, device=DEVICE,
                                llm_batch=LLM_BATCH, **kw)

## 7. 彩排（几篇文档，3 到 5 分钟）
把整条流程走一遍。这里的分数没有意义，只是确认每一步都能跑通。

In [ ]:
import shutil, time
shutil.rmtree("/content/rehearsal", ignore_errors=True)
t0 = time.time()
run(TRAIN_DOCS[:6], DEV_DOCS[:3], "/content/rehearsal", f"{CACHE}/rehearsal", k_values=(5, 10), label_free_k=10,
    training_data="rehearsal (6 docs)")
minutes = (time.time() - t0) / 60
print(f"\nREHEARSAL OK in {minutes:.1f} min")

## 8. 正式运行（`train_debug` 300 篇 → `dev_debug` 100 篇）
每算完一块就存一次。如果断线了：重新连接后再点一次“全部运行”，会从断的地方接着跑。

In [ ]:
if ONLY_REHEARSAL:
    raise SystemExit("ONLY_REHEARSAL = True: stopped after the rehearsal")
OUT = "/content/results_llm_redocred"
shutil.rmtree(OUT, ignore_errors=True)
t0 = time.time()
SETTINGS = run(TRAIN_DOCS, DEV_DOCS, OUT, f"{CACHE}/full", k_values=tuple(K_VALUES))
print(f"\nDONE in {(time.time() - t0) / 60:.0f} min")

## 9. 结果（全部由仓库的 `scripts/evaluate.py` 打分）

读法：`F1` 是主要指标；`Detect F1` 看"这对实体有没有关系"判断得准不准；`Typing F1` 看"关系是哪一种"判断得准不准。`td-bert-*` 两行就是 Laurent 的方法在同样小规模下的成绩，其他行和它比。

参考：Laurent 用全部 3,053 篇训练时，在 `dev_debug` 上 M2 = 48.45，M1 = 43.21；他只用 10% 的文档（305 篇）时，在整个 dev 上 M2 = 42.9，M1 = 38.9。

In [ ]:
import pandas as pd
board = pd.read_csv(f"{OUT}/leaderboard.csv")
cols = ["micro_f1", "ign_f1", "micro_p", "micro_r", "detect_f1", "typing_f1", "macro_f1", "graph_f1"]
table = board[["run", "supervision"] + cols].copy()
table[cols] = (100 * table[cols]).round(2)
table = table.rename(columns={"micro_f1": "F1", "ign_f1": "Ign F1", "micro_p": "P", "micro_r": "R",
                              "detect_f1": "Detect F1", "typing_f1": "Typing F1", "macro_f1": "Macro F1",
                              "graph_f1": "Graph F1"}).sort_values("F1", ascending=False)
pd.set_option("display.width", 250)
print(table.to_string(index=False))
print()
print(json.dumps(SETTINGS, indent=1)[:6000])

## 10. 打包下载
把 `results_llm_redocred.zip` 发给我。里面有每个方法的预测文件、分数、LLM 的原始回答，不包含密钥。

In [ ]:
shutil.make_archive("/content/results_llm_redocred", "zip", OUT)
if CACHE.startswith("/content/drive"):
    shutil.copy("/content/results_llm_redocred.zip", f"{CACHE}/results_llm_redocred.zip")
    print("copy kept on Google Drive:", f"{CACHE}/results_llm_redocred.zip")
try:
    from google.colab import files
    files.download("/content/results_llm_redocred.zip")
except Exception as e:
    print("download it from the Files panel on the left:", e)